# Génération de format de chargement de tarifs

Génération de deux fichiers parquet pour les tarifs et pour les liens tarif-pdc.

In [1]:
from datetime import datetime, timezone
import sys
from pathlib import Path
import json
import pandas as pd
# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))
from source.tariff import TariffObject, TariffDimensionTypeEnum, TariffElement, TariffElements, PriceComponent

In [2]:
statiques = pd.read_csv("../data/qualicharge/donnees_statiques___liste_stations_pdcs_puissance_2026-06-06T23_40.csv", sep=",")
total_50 = len(statiques[statiques["puissance_nominale"] >= 50])
total_50

37157

In [3]:
def tariffs_to_dataframe(tariffs: list[dict], schema: dict, verbose: bool = False) -> pd.DataFrame:
    original_id = []
    original_last_updated = []
    raw = []
    start= []
    end = []
    for tarif in tariffs:
        assert TariffObject.is_valid_json(schema, tarif, verbose=verbose), f"Tariff {tarif['id']} is not valid according to the schema"
        original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
        original_last_updated.append(tarif["last_updated"])
        raw.append(tarif)
        start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime(1970, 1, 1, tzinfo=timezone.utc).isoformat()))))
        end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
    return pd.DataFrame({
        "original_id": original_id,
        "original_last_updated": original_last_updated,
        "raw": raw,
        "start": start,
        "end": end})

In [4]:
def tariffs_from_OCPI(locations: list, tariffs: list, schema: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    tariff = tariffs_to_dataframe(tariffs, schema)  
    id_pdc_itinerance = []
    id_tariff = []
    for location in locations:
        for evse in location["evses"]:
            id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
            tariff_id = location["country_code"] + location["party_id"]
            tariff_id+= evse["connectors"][0]["tariff_ids"][0]
            if tariff_id not in tariff["original_id"].values:
                print(f"Tariff ID {tariff_id} not found in original tariffs")
            id_tariff.append(tariff_id)
    tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": id_pdc_itinerance,
        "id_tariff": id_tariff})
    return (tariff, tariffpdc)

In [5]:
def tariff_simple(country_code: str, party_id: str, id: str, last_updated: str, dimensions: list[str], prices: list[float], start_date_time: str=None, end_date_time: str=None):
    price_components = []
    for i in range(len(dimensions)):
        price_component = PriceComponent(type=TariffDimensionTypeEnum(dimensions[i]), price=prices[i], step_size=1)
        price_components.append(price_component)
    tariff_elements = TariffElements(
        root=[TariffElement(price_components=price_components)]
    )
    tariff = TariffObject(
        country_code=country_code,
        party_id=party_id,
        id=id,
        elements=tariff_elements,
        last_updated=last_updated,
        start_date_time=start_date_time,
        end_date_time=end_date_time
    )
    return json.loads(tariff.model_dump_json(exclude_none=True))

In [6]:
def tariffs_to_text(tariffs: pd.DataFrame, operator: str) -> None:
    text = "# Tarifs au format texte des tarifs " + operator + "\n\n"
    for tarif in tariffs["raw"]:
        tariff = TariffObject.model_validate(tarif)
        text += tariff.to_text() + "\n"
    with open(f"../data/{operator}/tariffs_text.md", "w", encoding="utf-8") as f:
        f.write(text)

In [7]:
def export_tariffs(tariffs: pd.DataFrame, tariffpdc: pd.DataFrame, operator: str) -> None:
    print(f"len tariff, tariffpdc et % : {len(tariffs)}, {len(tariffpdc)}, {len(tariffpdc)/total_50*100:.2f} %")
    tariffs_to_text(tariffs, operator)
    tariffs.to_parquet(f"../data/{operator}/qualicharge_tariff.parquet", index=False)
    tariffpdc.to_parquet(f"../data/{operator}/qualicharge_tariffpdc.parquet", index=False)

## Tesla

In [ ]:
operator = "tesla"

with open(f'../data/{operator}/FR_Supercharging_locations.json', 'r', encoding='utf-8') as f:
    tesla_locations = json.load(f)
with open(f'../data/{operator}/FR_Supercharging_NTSLA_tariffs.json', 'r', encoding='utf-8') as f:
    tesla_tariffs = json.load(f)
with open("../source/schema.json") as f:
    schema = json.load(f)
tariffs, tariffpdc = tariffs_from_OCPI(tesla_locations["data"], tesla_tariffs["data"], schema)

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc et % : 262, 3898, 10.49 %


In [9]:
tariffpdc

,id_pdc_itinerance,id_tariff
0,FRTSLEA4QY2C,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
1,FRTSLEA4T0JE,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
2,FRTSLEA6K9GP,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
3,FRTSLEA6LMC5,NLTSL5fa1c37d-9aa9-4778-805b-8a1c1299765d
4,FRTSLE0003DB,NLTSL80791b33-aedd-4d7d-a0b8-1d831c44de68
...,...,...
3893,FRTSLECM5I9O,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3894,FRTSLECM5KYZ,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3895,FRTSLECM5QVJ,NLTSL64395007-0c2e-45bf-a948-1148c3a15815
3896,FRTSLECM5QVG,NLTSL64395007-0c2e-45bf-a948-1148c3a15815


## Fastned

In [ ]:
operator = "fastned"

with open("../source/schema.json") as f:
    schema = json.load(f)
tariff = tariff_simple(
    country_code="FR",
    party_id="FAS",
    id="tarif_unique_1",
    last_updated="2024-06-01T12:00:00Z",
    dimensions=["ENERGY"],
    prices=[0.5083]) # 0.61€/kWh TTC -> 0.61/1.2 = 0.5083€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_fastned = statiques[statiques["id_pdc_itinerance"].str.startswith("FRFAS", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_fastned["id_pdc_itinerance"].tolist(),
        "id_tariff": ["FRFAStarif_unique_1"] * len(statiques_fastned)})

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc et % : 1, 457, 1.23 %


In [11]:
print(tariffs["raw"][0], "\n")
print(tariffs, "\n")
print(tariffpdc)


{'country_code': 'FR', 'party_id': 'FAS', 'id': 'tarif_unique_1', 'elements': [{'price_components': [{'type': 'ENERGY', 'price': 0.5083, 'step_size': 1}]}], 'last_updated': '2024-06-01T12:00:00Z'} 

           original_id original_last_updated  \
0  FRFAStarif_unique_1  2024-06-01T12:00:00Z   

                                                 raw  \
0  {'country_code': 'FR', 'party_id': 'FAS', 'id'...   

                      start   end  
0 2024-06-01 12:00:00+00:00  None   

    id_pdc_itinerance            id_tariff
0       FRFASE3323310  FRFAStarif_unique_1
1       FRFASE3327905  FRFAStarif_unique_1
2       FRFASE3310001  FRFAStarif_unique_1
3       FRFASE3322903  FRFAStarif_unique_1
4       FRFASE3309605  FRFAStarif_unique_1
..                ...                  ...
452     FRFASE3300801  FRFAStarif_unique_1
453     FRFASE3311003  FRFAStarif_unique_1
454     FRFASE3332104  FRFAStarif_unique_1
455     FRFASE3302702  FRFAStarif_unique_1
456     FRFASE3321102  FRFAStarif_unique_1



## Atlante

In [ ]:
operator = "atlante"

tariffs_pdc = pd.read_csv(f"../data/{operator}/current_tariffs.csv", sep=",") # .rename(columns={"rateid": "id_tariff"})
tariffs_pdc["id_tariff"] = tariffs_pdc["rateid"].apply(lambda x: "FRATL" + x)
tariffpdc = tariffs_pdc[["id_pdc_itinerance", "id_tariff"]]

tariffs = tariffs_pdc.drop_duplicates(subset=["id_tariff"]).reset_index(drop=True)
tariffs["original_id"] = tariffs["id_tariff"]
tariffs["original_last_updated"] = tariffs["rate_date_modified"]
tariffs["raw"] = tariffs.apply(lambda row: tariff_simple(
    country_code="FR",
    party_id="ATL",
    id=row["rateid"],
    last_updated=row["rate_date_modified"] + "Z",
    dimensions=["ENERGY", "TIME", "PARKING_TIME"],
    prices=[row["costkwh"]/1.2, row["costchargingtime"]/1.2, row["costparkingtime"]/1.2]), axis=1)
tariffs["start"] = tariffs["rate_date_modified"]
tariffs["end"] = None
tariffs = tariffs[["original_id", "original_last_updated", "raw", "start", "end"]]

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc et % : 9, 1392, 3.75 %


In [13]:
print(tariffs["raw"][0], "\n")
print(tariffs, "\n")
print(tariffpdc)

{'country_code': 'FR', 'party_id': 'ATL', 'id': '5152fed1-8794-4df8-9bed-55deef54d319', 'elements': [{'price_components': [{'type': 'ENERGY', 'price': 0.49166666666666664, 'step_size': 1}, {'type': 'TIME', 'price': 0.0, 'step_size': 1}, {'type': 'PARKING_TIME', 'price': 0.0, 'step_size': 1}]}], 'last_updated': '2026-04-01T16:41:07Z'} 

                                 original_id    original_last_updated  \
0  FRATL5152fed1-8794-4df8-9bed-55deef54d319  2026-04-01 16:41:07.000   
1  FRATL99d24794-f778-46e6-8171-555bab3a25c4  2026-04-01 16:39:45.000   
2  FRATL34f7e4fd-ea01-402e-95ed-1a92fedd4a39  2026-04-01 16:46:11.000   
3  FRATL6fda9486-bb45-49e1-9b86-84b8f251a478  2026-04-01 16:40:14.000   
4  FRATL0d03b682-293f-4f8a-8444-35641b04c757  2026-04-01 16:43:31.000   
5  FRATLe06972d7-a9e5-45f3-b044-5a6c712110b2  2026-04-01 16:38:57.000   
6  FRATL30d42725-886e-4aaf-a2b6-4fa1d232bb4b  2026-04-01 16:42:30.000   
7  FRATL7263d104-13cf-4838-9a5a-2974f967fb7a  2026-04-01 16:37:23.000   
8  FR

## Total

In [14]:

with open('../data/total/Tarifs.json', 'r', encoding='utf-8') as f:
    total_tarifs = json.load(f)
len(total_tarifs)

65

## Driveco

In [15]:

with open('../data/driveco/Tarifs.json', 'r', encoding='utf-8') as f:
    driveco_tarifs = json.load(f)
len(driveco_tarifs)

85